# Augmentation v2 — deterministic rung 1, the supply gate, and the synthetic rung

Three things changed, and this notebook runs all of them:

1. **Rung 1 stopped calling the model.** `decorate`, `inject`,
   `operator_syntax_rewrite` and `stat_rewrite`'s cut all implement `apply()`
   and are served by a local re-measure. Only expansion still buys a
   completion.
2. **A claimability filter on the supply.** Mined surfaces outlive the bank
   that claimed them; a repair silently rots the artifact and nothing noticed.
3. **The gate is a fork.** What rung 1 cannot reach is routed to a synthetic
   rung that writes the query *and* the document answering it.

Nothing below spends a completion unless the last section is run deliberately.

In [1]:
import pandas as pd

from augmentation.campaign import AugmentationCampaign
from augmentation.config import AugmentationPaths
from augmentation.loop import AugmentationLoop
from augmentation.parents import ParentPool
from composition.cells import CELLS, CELLS_BY_NAME
from dataset_registry import DATASETS

pd.set_option("display.width", 220, "display.max_columns", None)
paths = AugmentationPaths()
catalog = pd.read_parquet(paths.catalog).astype({"query_id": str})
selection = pd.read_parquet(
    paths.data_dir / "composition/cell_selection.parquet"
).astype({"query_id": str})
parents = ParentPool(catalog, selection, {d.name: d for d in DATASETS})
loop = AugmentationLoop(selection, sheet_path=paths.cell_order_sheet, parents=parents)

/Users/andrei/projects/hybrid-search-rrf-dataset/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1 · Every cell still has its `looks_like`

The only field a generator ever sees — and now the synthetic rung's prompt is
built from it, so a missing one is a cell that cannot be generated.

In [2]:
hungry = set(pd.read_parquet(paths.cell_order_sheet)["floor"])
audit = pd.DataFrame([
    {"cell": c.name, "hungry": c.name in hungry, "looks_like": c.looks_like}
    for c in CELLS
]).assign(written=lambda d: d["looks_like"].notna())
print(f"{audit['written'].sum()} of {len(audit)} cells have looks_like")
print(f"hungry cells still missing one: {(audit['hungry'] & ~audit['written']).sum()}")

44 of 44 cells have looks_like
hungry cells still missing one: 0


## 2 · Rung 1 without a model

Each family's `apply()` on one query. `None` means the family defers — the row
goes to the LLM, which is the routing decision made per row rather than
guessed per family.

In [3]:
from augmentation.operators import (
    DecorateOperator, InjectOperator, OperatorSyntaxRewrite, StatRewrite,
)

Q = "how to configure the nginx reverse proxy for a docker container"
rows = [
    ("decorate", DecorateOperator(), "marker:politeness",
     pd.Series({"query_id": "q1", "query": Q, "floors": []})),
    ("inject", InjectOperator(), "code_symbol_named_in_prose",
     pd.Series({"query_id": "q1", "query": Q, "floors": [],
                "surfaces": ("cPGES",), "bank": "code_identifier"})),
    ("inject (keyword-gated bank)", InjectOperator(), "version_pinned_technical",
     pd.Series({"query_id": "q1", "query": Q, "floors": [],
                "surfaces": ("17.2",), "bank": "version_string"})),
    ("operator_syntax", OperatorSyntaxRewrite(), "logical:operator_syntax",
     pd.Series({"query_id": "q1", "query": "python and rust benchmarks",
                "floors": []})),
    ("stat_rewrite (cut)", StatRewrite(), "version_pinned_technical",
     pd.Series({"query_id": "q1", "query": Q, "floors": [], "surfaces": (),
                "stat_value": 11.0, "gold_text": "nginx proxy docker"})),
]
pd.DataFrame([
    {"family": name,
     "result": op.apply(parent, floor, str(parent["query"])) or "— defers to the model"}
    for name, op, floor, parent in rows
]).set_index("family")

,result
family,
decorate,"would you, how to configure the nginx reverse ..."
inject,how to configure the nginx reverse proxy for a...
inject (keyword-gated bank),— defers to the model
operator_syntax,python AND rust benchmarks
stat_rewrite (cut),how configure nginx reverse proxy for a docker...


**`inject` on a keyword-gated bank defers.** `version_string` only claims
`17.2` next to a gate word (`SPSS version 22.0` is a declared positive,
`p = 0.05` a declared negative). Appending it bare leaves the target unmet, so
the row reaches the model instead of dropping against a target no placement of
ours can satisfy. Containment and claimability are different properties.

## 3 · How much of the work is still bought

A dry census over every hungry cell: how many calls route deterministically
and how many reach the model. No completion is spent — `_deterministic` is
asked what it *would* do.

In [4]:
from augmentation.dispatch import calls_for

PER_CELL = 15
rows = []
for floor in loop.order_sheet()["floor"]:
    cell = CELLS_BY_NAME.get(floor)
    if cell is None:
        continue
    try:
        result, pool_df = loop.demand(floor)
    except ValueError:
        rows.append({"cell": floor, "note": "no servable parent"})
        continue
    free = paid = 0
    paid_by = set()
    for record in pool_df.head(PER_CELL).to_dict("records"):
        parent = loop.grounded(result, pd.Series(record))
        text = str(parent["query"])
        for call in calls_for(result, cell, parent):
            produced = AugmentationLoop._deterministic(floor, call, parent, text)
            if produced is None:
                paid += 1
                paid_by |= {o.declaration.operator for o in call.operators}
            else:
                free, text = free + 1, produced
    rows.append({"cell": floor, "parents": len(pool_df), "free": free,
                 "paid": paid, "paid_by": ", ".join(sorted(paid_by)) or "-"})
census = pd.DataFrame(rows).fillna(0)
print(f"free {int(census['free'].sum())} calls | paid {int(census['paid'].sum())} calls")
census

[trec-dl-2022] supply: dropped 32/222 rows (14.4%) their bank no longer claims


[crumb-code-retrieval] supply: dropped 49/196 rows (25.0%) their bank no longer claims
[crumb-set-operation-entity-retrieval] supply: dropped 1/1 rows (100.0%) their bank no longer claims — mostly datetime


[crumb-code-retrieval] supply: dropped 18/140,959 rows (0.0%) their bank no longer claims


[lotte-technology-search] supply: dropped 1/722 rows (0.1%) their bank no longer claims


[crumb-clinical-trial] supply: dropped 13/488 rows (2.7%) their bank no longer claims


...

[crumb-paper-retrieval] supply: dropped 1,343/1,346 rows (99.8%) their bank no longer claims — mostly version_string
[crumb-set-operation-entity-retrieval] supply: dropped 943/944 rows (99.9%) their bank no longer claims — mostly version_string
[crumb-stack-exchange] supply: dropped 211/212 rows (99.5%) their bank no longer claims — mostly version_string
[crumb-theorem-retrieval] supply: dropped 7/7 rows (100.0%) their bank no longer claims — mostly version_string
[crumb-tip-of-the-tongue] supply: dropped 101/101 rows (100.0%) their bank no longer claims — mostly version_string


[crumb-code-retrieval] supply: dropped 18/140,959 rows (0.0%) their bank no longer claims


[lotte-technology-search] supply: dropped 1/722 rows (0.1%) their bank no longer claims


free 370 calls | paid 0 calls


,cell,parents,free,paid,paid_by
0,legal_citation_canonical,1,2,0,-
1,datetime_token_present,54,15,0,-
2,single_token_char_blob,150709,15,0,-
3,business_temporal_reference,2,2,0,-
4,symbol_pile_no_grammar,16,30,0,-
5,bibliographic_catalog_identifier,92,30,0,-
6,bio_clinical_identifier,182,30,0,-
7,travel_transport_code,6,6,0,-
8,standards_compliance_lookup,20,15,0,-
9,boolean_operator_query,5172,15,0,-


## 4 · The supply filter

`surfaces.parquet` records spans as the banks claimed them *at mining time*.
`version_string` was repaired afterwards, so most of its mined supply is now
unclaimable — and eligibility was faithfully selecting it. The filter drops
what can no longer serve its own target, and names the rotten bank.

In [5]:
from augmentation.supply import SupplyIndex, lane_dirs

index = SupplyIndex(paths)
report = []
for lane in ("beir-nfcorpus", "crumb-stack-exchange", "bright-leetcode", "rarb-code"):
    mined = index.load(lane)
    if mined.empty:
        continue
    kept = index.claimable(mined, lane)
    report.append({"lane": lane, "mined": len(mined), "kept": len(kept),
                   "kept_%": round(100 * len(kept) / len(mined), 1)})
pd.DataFrame(report).set_index("lane")

[beir-nfcorpus] supply: dropped 12,951/44,301 rows (29.2%) their bank no longer claims — mostly version_string


[crumb-stack-exchange] supply: dropped 19,913/190,752 rows (10.4%) their bank no longer claims — mostly version_string


[bright-leetcode] supply: dropped 1,844/107,163 rows (1.7%) their bank no longer claims — mostly version_string


[rarb-code] supply: dropped 14,884/620,869 rows (2.4%) their bank no longer claims — mostly version_string


,mined,kept,kept_%
lane,,,
beir-nfcorpus,44301,31350,70.8
crumb-stack-exchange,190752,170839,89.6
bright-leetcode,107163,105319,98.3
rarb-code,620869,605985,97.6


## 5 · The gate: what rung 1 reaches, and what it does not

`CellPlan.reachable` is **not** the parent count — it is zero whenever a band
is unserved, because those rows land in whatever cell they measure into rather
than the one requested. The plan splits each cell's demand accordingly.

In [6]:
plan = AugmentationCampaign(loop).plan()
plan[["floor", "missing", "target_rows", "synthetic_rows", "source_dataset", "action"]]

[trec-dl-2022] supply: dropped 32/222 rows (14.4%) their bank no longer claims


[crumb-code-retrieval] supply: dropped 49/196 rows (25.0%) their bank no longer claims
[crumb-set-operation-entity-retrieval] supply: dropped 1/1 rows (100.0%) their bank no longer claims — mostly datetime


[crumb-code-retrieval] supply: dropped 18/140,959 rows (0.0%) their bank no longer claims


[lotte-technology-search] supply: dropped 1/722 rows (0.1%) their bank no longer claims


[crumb-clinical-trial] supply: dropped 13/488 rows (2.7%) their bank no longer claims


...

[crumb-set-operation-entity-retrieval] supply: dropped 943/944 rows (99.9%) their bank no longer claims — mostly version_string
[crumb-stack-exchange] supply: dropped 211/212 rows (99.5%) their bank no longer claims — mostly version_string
[crumb-theorem-retrieval] supply: dropped 7/7 rows (100.0%) their bank no longer claims — mostly version_string
[crumb-tip-of-the-tongue] supply: dropped 101/101 rows (100.0%) their bank no longer claims — mostly version_string


[crumb-code-retrieval] supply: dropped 18/140,959 rows (0.0%) their bank no longer claims


[lotte-technology-search] supply: dropped 1/722 rows (0.1%) their bank no longer claims


campaign plan — 22 hungry floors, 241 target rows from parents, 42 needing the synthetic rung:
                           floor  missing                operator              gate             action  target_rows  synthetic_rows         source_dataset
        legal_citation_canonical    400.0    inject, stat_rewrite    coherence_gate skip: pilot staged            0               0                  orcas
          datetime_token_present    399.0    inject, stat_rewrite    coherence_gate            produce           12               0   freshstack-langchain
          single_token_char_blob    398.0            stat_rewrite declaration_audit            produce           22               0         scirgen-geo-en
     business_temporal_reference    398.0    inject, stat_rewrite    coherence_gate            produce            2              23                  orcas
          symbol_pile_no_grammar    398.0    inject, stat_rewrite    coherence_gate skip: pilot staged            0               

,floor,missing,target_rows,synthetic_rows,source_dataset,action
0,legal_citation_canonical,400.0,0,0,orcas,skip: pilot staged
1,datetime_token_present,399.0,12,0,freshstack-langchain,produce
2,single_token_char_blob,398.0,22,0,scirgen-geo-en,produce
3,business_temporal_reference,398.0,2,23,orcas,produce
4,symbol_pile_no_grammar,398.0,0,0,freshstack-angular,skip: pilot staged
5,bibliographic_catalog_identifier,398.0,0,0,orcas,skip: pilot staged
6,bio_clinical_identifier,398.0,0,0,crumb-legal-qa,skip: pilot staged
7,travel_transport_code,397.0,6,19,crumb-clinical-trial,produce
8,standards_compliance_lookup,394.0,9,0,crumb-clinical-trial,produce
9,boolean_operator_query,393.0,28,0,scirgen-geo-en,produce


`source_dataset` is derived, not configured: the lane the cell's own parents
come from, so the constructed collection borrows topically plausible
distractors. A cell with no parents at all gets none — the campaign reports it
rather than inventing a lane.

## 6 · The synthetic rung

No parent and no corpus surface, so the query is written from the cell's own
bands. The prompt is built from the predicate plus `looks_like`, and each span
band is shown shapes its detector actually accepts.

In [7]:
from augmentation.synthetic import SyntheticOperator

synth = SyntheticOperator()
print(synth.instruction("symbol_pile_no_grammar", pd.Series({"branch_index": 0})))

Write ONE realistic search query somebody would actually type. It must satisfy every one of these measured properties:
- contains at least 2 code_identifier (shapes that count: r1_nqyyxve_c_lsiaf_rb_krzm_ce4_ppoj, VRGWG8B_VS_QF1VSZ_N5DR_WZRETYAE, RH_RZ51D_J)
- natural_language_share under 0.1
- length_words under 10
A query of this kind looks like: Two or more bare technical tokens sitting side by side with no connective words at all — like a pasted stack-trace fragment or a list of symbols someone dropped into a search box. Not a question, not a sentence: no "how", "what", "the", "of", "in", "and".

Reply with the query text only.


### End to end, with the model stubbed

Two calls per row — query first, so a query that misses its bands costs one
completion instead of two. The stub returns a fixed pair, which exercises
every check between the completion and the banked row without spending.

In [8]:
from query_taxonomy.features import FeatureExtractor
from taxonomy_generators.verify import verify

from augmentation.constructed import ConstructedDocs
from augmentation.engine import AugmentationOutcome
from augmentation.pool import GeneratedPool
from augmentation.qrels import AugmentationQrels

class ScriptedPair:
    """Query on the first call, document on the second."""
    def __init__(self, query, doc):
        self.query, self.doc, self.calls = query, doc, []
        self._x = FeatureExtractor(engines=None)
    def run(self, instruction, prompt, targets, *, tool_loop=False):
        self.calls.append(instruction)
        text = self.doc if prompt.startswith("Query:") else self.query
        report = verify(text, targets, extractor=self._x)
        return AugmentationOutcome(text=text, accepted=report.passed,
                                   attempts=1, checks=report.checks)

throwaway = AugmentationPaths(data_dir="/tmp/smoke-synth")
stub = ScriptedPair("cPGES lipoxinA4",
                    "cPGES and lipoxinA4 are lipid mediators in inflammation.")
dry = AugmentationLoop(
    selection, sheet_path=paths.cell_order_sheet, parents=parents, engine=stub,
    pool=GeneratedPool(throwaway), qrels=AugmentationQrels(throwaway),
    docs=ConstructedDocs(throwaway),
)
made = dry.synthesize("symbol_pile_no_grammar", 1, source_dataset="beir-nfcorpus")
print("engine calls:", len(stub.calls), "(query, then document)")
made[["query_id", "query", "operator", "provenance", "generated_from",
      "answer_key", "grounding_doc_id"]]

synthesize:symbol_pile_no_grammar:   0%|          | 0/1 [00:00<?, ?row/s]

synthesize:symbol_pile_no_grammar: 100%|██████████| 1/1 [00:00<00:00,  2.22row/s]

synthesize:symbol_pile_no_grammar: 100%|██████████| 1/1 [00:00<00:00,  2.22row/s]

synthesize:symbol_pile_no_grammar: 100%|██████████| 1/1 [00:00<00:00,  2.22row/s]

+ syn-symbol_pile_no_grammar-1: 'cPGES lipoxinA4'
  0 hops, 0 tokens, 0.5s wall (0.0s of it LLM) -> 0.0 hops/row, 0 tokens/row, 0.5s/row
engine calls: 2 (query, then document)


,query_id,query,operator,provenance,generated_from,answer_key,grounding_doc_id
0,syn-symbol_pile_no_grammar-1,cPGES lipoxinA4,synthesize,synthetic,,minted,constructed-syn-symbol_pile_no_grammar-1


In [9]:
print("constructed doc:")
display(dry.docs.load())
print("answer key — depth one, discrimination comes from borrowed distractors:")
display(dry.qrels.load())

constructed doc:


,doc_id,source_dataset,for_query,text
0,constructed-syn-symbol_pile_no_grammar-0,beir-nfcorpus,syn-symbol_pile_no_grammar-0,cPGES and lipoxinA4 are lipid mediators in inf...
1,constructed-syn-symbol_pile_no_grammar-1,beir-nfcorpus,syn-symbol_pile_no_grammar-1,cPGES and lipoxinA4 are lipid mediators in inf...


answer key — depth one, discrimination comes from borrowed distractors:


,query_id,doc_id,relevance,source,inherited_from
0,syn-symbol_pile_no_grammar-0,constructed-syn-symbol_pile_no_grammar-0,1,constructed,None
1,syn-symbol_pile_no_grammar-1,constructed-syn-symbol_pile_no_grammar-1,1,constructed,None


## 7 · What this does not show

- **No completion has been spent here.** The synthetic rung is verified
  structurally — plumbing, idempotency, routing — never for output quality.
  Run `campaign.run()` below to find out whether the band instructions land.
- **The example surfaces are poor.** `code_identifier` reverse-samples as
  `r1_nqyyxve_c_lsiaf_rb_krzm_ce4_ppoj`: valid to the bank, absurd to a reader.
  `taxonomy_generators/overrides.py` exists for exactly this and is empty.
- **Rung 1 can still produce nonsense.** `symbol_pile_no_grammar` cutting a
  Russian programming problem down to `.,,,,,.,. — (, ). k,.,.,.` satisfies
  every band and is not a query anybody would type. That is what
  `credit_gate=coherence_gate` exists to catch, and the audit is unrun.

In [10]:
# LIVE — spends completions. The deterministic families cost nothing; the bill
# is stat_rewrite's expansion plus the synthetic rung's two calls per row.
#
# import contextlib
# from pathlib import Path
#
# log = Path("/tmp/campaign_run.log")
# with log.open("w") as f, contextlib.redirect_stdout(f):
#     summary = AugmentationCampaign(loop).run()
# summary[["floor", "accepted", "synthesized", "action"]]